# South Movie · continue after Ombatthane Dikku

The paused Kaggle run confirmed **1,227 of 2,727 movie photos**, repaired the earlier captions, and removed 12 old text messages. The last confirmed photo was **Ombatthane Dikku (2022)**, source 1567. You checked that it was the last post in **@southdhamal**. The next photo is **Annapoorani – The Goddess of Food (2023)**, source 1568. There are **1,500 photos left**, including Annapoorani.

1. In Kaggle, attach the **Output** of the version that reported **1,227 confirmed**. Its progress file is `south_movie_verified_checkpoint.json`. If you downloaded that single JSON, upload it as a private Kaggle input instead. A session that still has the checkpoint in `/kaggle/working` picks it up automatically.
2. Keep Internet on and the existing `TELEGRAM_BOT_TOKEN` secret enabled. Use **Save Version → Save & Run All** with one uploader running for the channel. The decision about this attempt at Annapoorani is already filled in.
3. At the end, keep the newest `south_movie_verified_checkpoint.json` from Output. If another upload disconnects, use the new recovery key printed alongside that photo. Each decision applies to a single delivery attempt.

**If you cannot find the checkpoint**, export the full, current **destination** channel in Telegram Desktop as JSON and add its `result.json` to Kaggle Input. Export all dates through the pause; the media files themselves are optional. The notebook verifies the complete sequence through Ombatthane Dikku and recovers the actual Telegram post IDs. It also recognizes Annapoorani if it appears immediately afterward. An older export or a mismatch pauses before sending.

The first live output shows the confirmed count and next movie. Use `RUN_MODE = "PREVIEW"` in the settings cell if you want to check the input without contacting Telegram.

In [ ]:
# Ready to continue the verified run in @southdhamal.
RUN_MODE = "REPAIR_AND_CONTINUE"
# You confirmed that Ombatthane Dikku was the last channel post.
# Integer 0 records that the next photo was absent for this specific attempt.
PENDING_KEY = "d5c39ccae9404bc69dc6fa79cd21b846"
PENDING_MESSAGE_ID = 0
CHECKPOINT_PATH = ""      # Optional path to the latest verified checkpoint JSON.
DESTINATION_EXPORT = ""   # Optional full current destination result.json.
SOURCE_ZIP = ""           # Optional pinned South Movie source archive.
RETRY_REVIEW_ITEMS = False

In [ ]:
from pathlib import Path, PurePosixPath
from urllib.parse import urlsplit
from collections import Counter
from contextlib import contextmanager
import base64
import hashlib
import io
import json
import os
import re
import tempfile
import time
import uuid
import zipfile
import zlib

REPO = "kawakaze430/south-movie"
REF = "a4dbc229fa5cc2563801bfc7ba7d416fb8b96138"
JSON_BLOB = "9d007411d3dfa271251c301633e82e279c96e0f9"
TARGET = "@southdhamal"
CHAT_ID = -1004376620825
LOG_BOT = "poste35dbot"
V1_DIGEST = "ee80be4a32337c3beb752264c3ee83cd4c96277cd84296ddadfbd9893d978024"
V2_DIGEST = "0494bf14f6d41332f78fb68f18765595e1c2c21554055d7622007bef8397d106"
INPUT = Path("/kaggle/input")
WORK = Path("/kaggle/working") if Path("/kaggle/working").is_dir() else Path.cwd() / "south_movie_output"
CACHE = Path("/kaggle/temp") if Path("/kaggle").is_dir() else Path(tempfile.gettempdir()) / "south_movie_cache"
CHECKPOINT_NAME = "south_movie_verified_checkpoint.json"
OLD_NAMES = ("progress_south_movie_southdhamal.json", "south_movie_clean_resume_checkpoint.json")
WORK.mkdir(parents=True, exist_ok=True)

class Pause(Exception):
    """Stop at a recoverable boundary and let Kaggle publish the run's outputs."""

def require(condition, message):
    if not condition:
        raise Pause(message)

def sha(data):
    return hashlib.sha256(data).hexdigest()

def write_json(path, value):
    path = Path(path)
    with tempfile.NamedTemporaryFile("w", encoding="utf-8", dir=path.parent, delete=False) as out:
        tmp = Path(out.name)
        try:
            json.dump(value, out, ensure_ascii=False, separators=(",", ":"))
            out.flush()
            os.fsync(out.fileno())
        except BaseException:
            tmp.unlink(missing_ok=True)
            raise
    try:
        os.replace(tmp, path)
    finally:
        tmp.unlink(missing_ok=True)

def read_json(path):
    try:
        return json.loads(Path(path).read_text(encoding="utf-8-sig"))
    except (ValueError, OSError) as exc:
        raise Pause(f"Cannot read JSON from {Path(path).name}. Supply the complete file. {type(exc).__name__}") from None

def plain(value):
    if isinstance(value, str):
        return value
    if isinstance(value, list):
        return "".join(plain(part) for part in value)
    if isinstance(value, dict):
        return plain(value.get("text", ""))
    return ""

def fingerprint(kind, caption):
    # Telegram can trim outside whitespace and normalize line endings during a send.
    return sha((kind + "\0" + caption.replace("\r\n", "\n").strip()).encode())

WEB_URL = re.compile(r"https?://[^\s<>]+", re.I)
def movie_url(url):
    p = urlsplit(url)
    host = (p.hostname or "").lower().rstrip(".")
    blocked = ("t.me", "telegram.me", "telegram.dog", "whatsapp.com", "wa.me",
               "instagram.com", "play.google.com", "m.media-amazon.com")
    return (p.scheme in ("https", "http") and bool(host)
            and not any(host == x or host.endswith("." + x) for x in blocked)
            and not (host.removeprefix("www.") == "terabox.com" and p.path == "/make-money-online"))

def source_links(msg):
    links = []
    for ent in msg["text_entities"]:
        if ent.get("type") in ("link", "text_link"):
            targets = [ent.get("href", ent["text"])]
        else:
            targets = WEB_URL.findall(ent.get("text", ""))
        links.extend(u for u in targets if movie_url(u))
    return links

def valid_mid(value):
    return type(value) is int and value > 0

In [ ]:
# The original export supplies verified IDs of the 248 historical deliveries only.
# These values help check a new destination export. They cannot start an upload.
BASELINE_COUNT = 1227
INTERRUPTED_SOURCE = "1568"
INTERRUPTED_ATTEMPT = "d5c39ccae9404bc69dc6fa79cd21b846"
HISTORICAL_DESTINATION_IDS = (3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99, 100, 101, 102, 103, 104, 105, 106, 107, 108, 109, 110, 111, 112, 113, 114, 115, 116, 117, 118, 119, 120, 121, 122, 123, 124, 125, 126, 127, 128, 129, 130, 131, 132, 133, 134, 135, 136, 137, 138, 139, 140, 141, 142, 143, 144, 145, 146, 147, 148, 149, 150, 151, 152, 153, 154, 155, 156, 157, 158, 159, 160, 161, 162, 163, 164, 165, 166, 167, 168, 169, 170, 171, 172, 173, 174, 175, 176, 177, 178, 179, 180, 181, 182, 183, 184, 185, 186, 187, 188, 189, 190, 191, 192, 193, 194, 195, 196, 197, 198, 199, 200, 201, 202, 203, 204, 205, 206, 207, 208, 209, 210, 211, 212, 213, 214, 215, 216, 217, 218, 219, 220, 221, 222, 223, 224, 225, 226, 227, 228, 229, 230, 231, 232, 233, 234, 235, 236, 237, 238, 239, 240, 241, 242, 243, 244, 245, 246, 247, 248, 249, 250)

### Verified source data

The folded cell contains compressed JSON with the cleaned captions, original photo checksums and legacy source IDs. The notebook reads it as data. The live run independently checks each selected caption’s movie URLs and each original photo against the pinned source.

In [ ]:
PLAN_SHA = "82d32838cd36209cd315985ae14dc0ec8bae3d44e18635c89820d725bec8ee97"
PACKED_PLAN = (
    "..."
)

In [ ]:
def unpack_plan():
    raw = zlib.decompress(base64.b64decode(PACKED_PLAN))
    require(sha(raw) == PLAN_SHA, "The embedded posting plan is damaged. Download this notebook again.")
    plan = json.loads(raw)
    require(plan["ref"] == REF, "The embedded plan belongs to a different source.")
    posts = plan["posts"]
    ids = [p["source_id"] for p in posts]
    require(ids == sorted(set(ids)), "Invalid source ordering or duplicate IDs in plan.")
    for p in posts:
        cap = p["caption"]
        links = WEB_URL.findall(cap)
        require(0 < len(cap.encode("utf-16-le")) // 2 <= 1024 and bool(links)
                and len(links) == p["link_count"] and all(movie_url(u) for u in links),
                f"Caption validation failed at source {p['source_id']}.")
    return plan

PLAN = unpack_plan()
POSTS = {str(p["source_id"]): p for p in PLAN["posts"]}
LEGACY = {str(sid): kind for sid, kind in PLAN["legacy_units"]}
PREFIX = [(str(sid), sig) for sid, sig in PLAN["failed_run_prefix"]]
FAILED_SID = PREFIX[-1][0]
EXCLUDED = PLAN["excluded"]
PLAN_COUNTS = dict(source_entries=PLAN["source_records"], movie_photos=len(POSTS),
                   movie_links=sum(p["link_count"] for p in POSTS.values()),
                   excluded=dict(Counter(EXCLUDED.values())))
write_json(WORK / "south_movie_clean_plan.json", {"counts": PLAN_COUNTS, "posts": PLAN["posts"]})
write_json(WORK / "south_movie_skipped_source_posts.json", EXCLUDED)
print(f"Source {REPO} @ {REF[:12]} | destination {TARGET} ({CHAT_ID})")
print(f"{len(POSTS):,} original movie photos • {PLAN_COUNTS['movie_links']:,} movie links • "
      f"{len(EXCLUDED):,} excluded source messages • 1 service entry")
print("Excluded by reason:", PLAN_COUNTS["excluded"])
require(list(POSTS)[BASELINE_COUNT - 1:BASELINE_COUNT + 1] == ["1567", INTERRUPTED_SOURCE], "Unexpected continuation boundary in source plan.")
print("\nNext photo at the logged pause • source 1568\n" + POSTS[INTERRUPTED_SOURCE]["caption"])

### Resume checks

The notebook matches the progress file to this channel, bot and source snapshot. It verifies that the first 1,227 movie photos already have confirmed deliveries and cleaned captions. Each new send saves a pending record before contacting Telegram and records the returned destination message ID on success.

In [ ]:
# Every message ID in receipts comes from a checkpoint, a matching export, or a confirmed API result.
def new_state(bot_id):
    return {"version": 3, "target": TARGET, "target_id": CHAT_ID, "bot_id": bot_id, "repo": REPO,
            "ref": REF, "plan_digest": PLAN_SHA, "receipts": {}, "done": {}, "issues": {},
            "pending": None, "resolutions": {}, "revision": 0}

def key_for(source_id):
    require(type(source_id) in (int, str) and str(source_id).isdigit(), "Invalid source ID in progress file.")
    return str(source_id)

def key_from_unit(unit):
    sid, sep, part = str(unit).partition(":")
    require(sep == ":" and part == "0" and sid in LEGACY, f"Unknown legacy unit {unit!r}.")
    return sid

def add_receipt(state, sid, msg_id, kind):
    require(sid in LEGACY and LEGACY[sid] == kind and valid_mid(msg_id), f"Invalid receipt at source {sid}.")
    row = {"message_id": msg_id, "kind": kind}
    prior = state["receipts"].get(sid)
    require(prior is None or prior == row, f"Conflicting Telegram message IDs at source {sid}.")
    state["receipts"][sid] = row

def set_legacy_receipts(state, rows):
    require(isinstance(rows, dict), "Legacy checkpoint has no sent-message map.")
    for key, row in rows.items():
        sid = key_from_unit(key)
        require(isinstance(row, dict) and key_for(row.get("source_id")) == sid, f"Receipt source mismatch for {key}.")
        add_receipt(state, sid, row.get("telegram_message_id"), row.get("kind"))

def check_identity(data, expected):
    for field, value in expected.items():
        require(data.get(field) == value, f"Checkpoint field {field!r} does not match this run.")

def import_pending(row, version):
    if row is None:
        return None
    require(isinstance(row, dict), "Invalid pending delivery in checkpoint.")
    sid = key_for(row.get("source_id"))
    require(row.get("part", 0) == 0 and LEGACY.get(sid) == row.get("kind"), "Unrecognized pending unit.")
    return {"sid": sid, "kind": row["kind"], "attempt": f"v{version}-{sid}"}

def normalize(data):
    require(isinstance(data, dict) and valid_mid(data.get("bot_id")), "Invalid bot identity in checkpoint.")
    version = data.get("version")
    s = new_state(data["bot_id"])
    if version == 1:
        check_identity(data, {"destination": TARGET, "target_id": CHAT_ID, "source_repo": REPO,
                              "source_ref": REF, "plan_sha256": V1_DIGEST})
        set_legacy_receipts(s, data.get("sent"))
        s["pending"] = import_pending(data.get("pending"), 1)
    elif version == 2:
        check_identity(data, {"target": TARGET, "target_id": CHAT_ID, "repo": REPO, "ref": REF,
                              "plan_digest": V2_DIGEST})
        set_legacy_receipts(s, data.get("legacy_sent"))
        require(isinstance(data.get("completed"), dict) and isinstance(data.get("repair_done"), dict),
                "The previous clean checkpoint is missing its completed or repair records.")
        for sid, row in data["completed"].items():
            require(isinstance(row, dict), f"Invalid clean receipt for source {sid}.")
            require(LEGACY.get(sid) == "photo", f"Invalid earlier completed source {sid}.")
            add_receipt(s, sid, row.get("telegram_message_id"), "photo")
        for action in data["repair_done"]:
            parts = action.split(":")
            if len(parts) == 3 and parts[1] == "remove" and parts[2] == "0":
                sid = parts[0]
                require(sid in s["receipts"] and LEGACY[sid] == "text", "Invalid legacy deletion record.")
                s["done"][sid] = "deleted"
            else:
                require(len(parts) == 2 and parts[1] == "caption" and parts[0] in s["receipts"]
                        and LEGACY[parts[0]] == "photo", f"Unknown previous repair action {action!r}.")
        s["pending"] = import_pending(data.get("send_pending"), 2)
    elif version == 3:
        check_identity(data, {"target": TARGET, "target_id": CHAT_ID, "repo": REPO, "ref": REF,
                              "plan_digest": PLAN_SHA})
        s = data
    else:
        raise Pause("Unrecognized checkpoint version. Attach the South Movie progress JSON.")
    # A pre-send snapshot at exactly the crash boundary must retain the uncertainty in the supplied log.
    if version < 3 and not s["pending"] and FAILED_SID not in s["receipts"]:
        s["pending"] = {"sid": FAILED_SID, "kind": "photo", "attempt": "v1-" + FAILED_SID}
    validate_state(s)
    return s

def validate_state(s):
    for field in ("receipts", "done", "issues", "resolutions"):
        require(isinstance(s.get(field), dict), f"Damaged checkpoint field {field!r}.")
    require(type(s.get("revision")) is int and s["revision"] >= 0, "Invalid checkpoint revision.")
    used = set()
    for sid, row in s["receipts"].items():
        require(isinstance(row, dict) and sid in LEGACY and row.get("kind") == LEGACY[sid]
                and valid_mid(row.get("message_id")), f"Invalid stored receipt for source {sid}.")
        mid = row["message_id"]
        require(mid not in used, "Two source posts point to the same Telegram message. Check the progress file.")
        used.add(mid)
    for sid, action in s["done"].items():
        row = s["receipts"].get(sid)
        correct = "deleted" if LEGACY.get(sid) == "text" else "caption"
        require(row and action == correct, f"Invalid recorded action at source {sid}.")
    for sid, issue in s["issues"].items():
        require(sid in LEGACY and isinstance(issue, dict) and isinstance(issue.get("reason"), str)
                and issue.get("operation") in ("sendPhoto", "editMessageCaption", "deleteMessage")
                and sid not in s["done"], f"Invalid review entry at source {sid}.")
        if issue["operation"] == "sendPhoto":
            require(sid in POSTS and sid not in s["receipts"], f"Inconsistent upload issue at {sid}.")
        else:
            require(sid in s["receipts"], f"Review entry has no Telegram receipt at {sid}.")
    for attempt, resolution in s["resolutions"].items():
        require(isinstance(attempt, str) and isinstance(resolution, dict)
                and resolution.get("sid") in LEGACY and type(resolution.get("message_id")) is int
                and resolution["message_id"] >= 0, "Invalid resolution in checkpoint.")
        if resolution["message_id"]:
            require(s["receipts"].get(resolution["sid"], {}).get("message_id") == resolution["message_id"],
                    "Resolved Telegram message ID conflicts with receipts.")
    pending = s.get("pending")
    if pending is not None:
        require(isinstance(pending, dict) and pending.get("sid") in LEGACY
                and pending.get("kind") == LEGACY[pending["sid"]] and isinstance(pending.get("attempt"), str)
                and pending["attempt"] and pending["attempt"] not in s["resolutions"]
                and pending["sid"] not in s["receipts"], "Pending send conflicts with the stored receipts.")

def covers(a, b):
    if a["bot_id"] != b["bot_id"]:
        return False
    for field in ("receipts", "done", "resolutions"):
        if any(a[field].get(k) != v for k, v in b[field].items()):
            return False
    for sid in b["issues"]:
        if (sid not in a["issues"] and sid not in a["done"]
                and not (a["pending"] and a["pending"]["sid"] == sid)
                and not any(item["sid"] == sid for item in a["resolutions"].values())):
            return False
    p = b["pending"]
    return not p or a["pending"] == p or p["sid"] in a["receipts"] or p["attempt"] in a["resolutions"]

def checkpoint_files():
    if CHECKPOINT_PATH.strip():
        f = Path(CHECKPOINT_PATH)
        require(f.is_file(), f"CHECKPOINT_PATH does not exist: {f}")
        supplied = [f]
    else:
        supplied = []
        # Kaggle may add a numeric suffix to an uploaded copy.
        names = {CHECKPOINT_NAME}
        if INPUT.exists():
            supplied = [f for f in INPUT.rglob("*.json")
                        if re.sub(r"\s*\(\d+\)(?=\.json$)", "", f.name) in names]
        supplied += [WORK / n for n in names if (WORK / n).is_file()]
    current = WORK / CHECKPOINT_NAME
    if current.exists():
        supplied.append(current)
    return sorted(set(p.resolve() for p in supplied))

def choose_state(named):
    require(named, "Attach south_movie_verified_checkpoint.json from the version with 1,227 confirmed photos, or attach a fresh destination result.json.")
    candidates = [(name, s) for name, s in named if all(covers(s, old) for _, old in named)]
    require(candidates, "The checkpoints disagree about a receipt or unresolved send. "
                        "Use the latest complete output in a fresh session and select it with CHECKPOINT_PATH.")
    name, s = max(candidates, key=lambda item: (item[1]["revision"], len(item[1]["receipts"]), bool(item[1]["pending"])))
    # The failed run durably recorded all of these before calling sendPhoto for 408.
    missing = [sid for sid, _ in PREFIX[:-1] if sid not in s["receipts"]]
    require(not missing, f"This progress file predates the failure. {len(missing)} of the first 248 "
                         "receipts are missing. Attach the failed version's Output or its matching destination export.")
    first_ids = [s["receipts"][sid]["message_id"] for sid, _ in PREFIX[:-1]]
    require(first_ids == sorted(first_ids), "The stored destination IDs for the logged run are out of order.")
    if FAILED_SID in s["receipts"]:
        require(s["receipts"][FAILED_SID]["message_id"] > first_ids[-1], "The destination ID for source 408 predates its neighbor.")
    require(FAILED_SID in s["receipts"] or (s["pending"] and s["pending"]["sid"] == FAILED_SID)
            or any(v["sid"] == FAILED_SID for v in s["resolutions"].values()),
            "The progress file does not account for the logged attempt at source 408.")
    print(f"Progress from {name} • {len(s['receipts'])} confirmed Telegram IDs.")
    return s

def state_from_export(data, bot_id):
    """Recover actual message IDs by checking the logged, repaired channel photo by photo."""
    require(isinstance(data, dict) and data.get("type") in ("public_channel", "private_channel")
            and str(data.get("id")) in (str(CHAT_ID), str(abs(CHAT_ID) - 10**12)),
            "Use a full Telegram Desktop export of the destination @southdhamal.")
    messages = data.get("messages")
    require(isinstance(messages, list) and all(isinstance(m, dict) for m in messages), "Invalid destination message list.")
    rows = [m for m in messages if m.get("type") == "message"]
    ids = [m.get("id") for m in rows]
    require(all(valid_mid(mid) for mid in ids) and ids == sorted(set(ids)), "Invalid IDs or ordering in destination export.")
    historic = dict(zip((sid for sid, _ in PREFIX[:-1]), HISTORICAL_DESTINATION_IDS))
    first_id = HISTORICAL_DESTINATION_IDS[0]
    # The setup marker predates the bot's first upload. Service entries have no movie caption.
    observed = [m for m in rows if m["id"] >= first_id]
    prior_photos = [sid for sid in historic if LEGACY[sid] == "photo"]
    prefix_photos = list(POSTS)[:BASELINE_COUNT]
    expected = prior_photos + [sid for sid in prefix_photos if sid not in historic]
    require(len(observed) in (len(expected), len(expected) + 1),
            "The destination export must extend through Ombatthane Dikku or the very next photo, Annapoorani. "
            "Use the latest progress JSON if the channel has moved on. The original 248-post export is too old.")
    has_interrupted = len(observed) == len(expected) + 1
    if has_interrupted:
        expected.append(INTERRUPTED_SOURCE)
    s = new_state(bot_id)
    present = set(ids)
    for sid, mid in historic.items():
        if LEGACY[sid] == "text":
            require(mid not in present, f"The destination export still contains previously removed text post {mid}.")
            add_receipt(s, sid, mid, "text")
            s["done"][sid] = "deleted"
    for sid, m in zip(expected, observed):
        mid = m["id"]
        require(sid not in historic or historic[sid] == mid, f"Historical Telegram ID differs at source {sid}.")
        require(m.get("photo") and plain(m.get("text", "")).replace("\r\n", "\n").strip() ==
                (POSTS[sid]["caption"].strip() if sid in POSTS else ""),
                f"The destination photo or caption differs at source {sid}. Attach the verified checkpoint.")
        add_receipt(s, sid, mid, "photo")
        s["done"][sid] = "caption"
    if has_interrupted:
        s["resolutions"][INTERRUPTED_ATTEMPT] = {"sid": INTERRUPTED_SOURCE,
                                                   "message_id": s["receipts"][INTERRUPTED_SOURCE]["message_id"]}
    else:
        s["pending"] = {"sid": INTERRUPTED_SOURCE, "kind": "photo", "attempt": INTERRUPTED_ATTEMPT}
    s["resolutions"]["v1-408"] = {"sid": "408", "message_id": 0}
    validate_state(s)
    return s

def persist(s):
    s["revision"] += 1
    write_json(WORK / CHECKPOINT_NAME, s)

def resolve_pending(s):
    p = s["pending"]
    if not p:
        return
    sid = p["sid"]
    unit = sid + ":0"
    post = POSTS.get(sid)
    if post:
        print("\nChecking pending source", unit, "\n" + post["caption"])
    decision_key = p["attempt"]
    print(f"Recovery key for this attempt: {decision_key}")
    raw_id = PENDING_MESSAGE_ID
    if isinstance(raw_id, str):
        match = re.fullmatch(r"https://t\.me/(?:southdhamal|c/4376620825)/([1-9][0-9]*)/?", raw_id.strip(), re.I)
        raw_id = int(match.group(1)) if match else None
    require(PENDING_KEY == decision_key and type(raw_id) is int and raw_id >= 0,
            f"Inspect {TARGET} for source {unit}. Set PENDING_KEY to {decision_key!r} for this attempt. "
            "Set PENDING_MESSAGE_ID to the destination post link or its numeric ID if present, "
            "or integer 0 only after checking it is absent. " +
            ("For 408 search 3gZCaC. The earlier sample uses a different set of links." if sid == FAILED_SID else ""))
    if raw_id:
        require(all(r["message_id"] != raw_id for r in s["receipts"].values()),
                "That Telegram message ID already belongs to a different receipt.")
        if sid == FAILED_SID:
            require(raw_id > max(s["receipts"][k]["message_id"] for k, _ in PREFIX[:-1]),
                    "That message predates source 408. Use the destination link for the full Vikram post.")
        add_receipt(s, sid, raw_id, p["kind"])
    s["resolutions"][p["attempt"]] = {"sid": sid, "message_id": raw_id}
    s["pending"] = None
    persist(s)

In [ ]:
class Rejected(Exception):
    """Telegram explicitly rejected an operation."""
    def __init__(self, code, description):
        self.code = code
        self.description = description
        super().__init__(f"Telegram {code}: {description}")

class UnknownSend(Pause):
    """Telegram may have accepted a photo. A receipt is needed before retry."""

def bot_token():
    value = os.environ.get("TELEGRAM_BOT_TOKEN", "").strip()
    if not value:
        try:
            from kaggle_secrets import UserSecretsClient
            value = (UserSecretsClient().get_secret("TELEGRAM_BOT_TOKEN") or "").strip()
        except Exception:
            pass
    require(re.fullmatch(r"[0-9]+:[A-Za-z0-9_-]+", value), "Enable the TELEGRAM_BOT_TOKEN secret for this Kaggle notebook.")
    return value

class Telegram:
    def __init__(self, token, session, wait=time.sleep):
        self.token = token
        self.session = session
        self.wait = wait
        self.last_mutation = 0.0

    def call(self, method, data=None, photo=None):
        import requests
        sending = method == "sendPhoto"
        mutation = sending or method in ("editMessageCaption", "deleteMessage")
        files = None
        if photo is not None:
            name, content = photo
            files = {"photo": (name, io.BytesIO(content), "image/jpeg")}
        for attempt in range(5):
            if mutation:
                gap = self.last_mutation + 3.2 - time.monotonic()
                if gap > 0:
                    self.wait(gap)
                self.last_mutation = time.monotonic()
            if files:
                files["photo"][1].seek(0)
            try:
                resp = self.session.post(f"https://api.telegram.org/bot{self.token}/{method}",
                                         data=data or {}, files=files, timeout=(15, 100), allow_redirects=False)
            except requests.RequestException:
                if sending:
                    raise UnknownSend("The connection ended during sendPhoto. Delivery is unknown.") from None
                if attempt == 4:
                    raise Pause(f"Telegram connection failed during {method}. Reattach the latest output and resume.") from None
                self.wait(2 ** attempt)
                continue
            try:
                body = resp.json()
            except (ValueError, TypeError):
                body = {}
            if not isinstance(body, dict):
                body = {}
            code = body.get("error_code")
            status = resp.status_code
            if status >= 500 or (type(code) is int and code >= 500):
                if sending:
                    raise UnknownSend(f"Telegram returned HTTP {status} during sendPhoto. Delivery is unknown.")
                if attempt == 4:
                    raise Pause(f"Telegram is unavailable during {method} (HTTP {status}). Resume later.")
                self.wait(2 ** attempt)
                continue
            if 200 <= status < 300 and body.get("ok") is True and "result" in body:
                return body["result"]
            if status == 429 or (body.get("ok") is False and code == 429):
                # An explicit 429 means this request was rejected, not delivered.
                # Telegram can request an 8-minute cooldown after large batches.
                # Wait and retry the SAME photo only when the server explicitly
                # rejected it; never auto-retry ambiguous timeouts/5xx.
                params = body.get("parameters")
                delay = params.get("retry_after", 30) if isinstance(params, dict) else 30
                delay = delay if type(delay) is int and delay >= 0 else 30
                if delay > 900 or attempt == 4:
                    raise Rejected(429, f"Rate limit. Resume after {delay + 1} seconds.")
                print(f"Telegram rate limit • waiting {delay + 1} seconds before retrying {method} (attempt {attempt + 2}/5).", flush=True)
                self.wait(delay + 1)
                continue
            if body.get("ok") is False and type(code) is int and 400 <= code < 500:
                desc = str(body.get("description", "Request rejected")).replace(self.token, "[redacted]")
                raise Rejected(code, desc[:250])
            if sending:
                raise UnknownSend(f"No usable sendPhoto receipt (HTTP {status}).")
            if attempt == 4:
                raise Pause(f"No usable Telegram response for {method} (HTTP {status}). Resume later.")
            self.wait(2 ** attempt)
        raise Pause(f"Telegram response unavailable for {method}.")

def verify_target(api):
    bot = api.call("getMe")
    chat = api.call("getChat", {"chat_id": TARGET})
    require(isinstance(chat, dict) and chat.get("type") == "channel" and chat.get("id") == CHAT_ID
            and str(chat.get("username", "")).lower() == TARGET[1:] and chat.get("title", "").strip() == "South Movie",
            "Destination verification failed. This notebook is tied to the South Movie channel in the run log.")
    require(isinstance(bot, dict) and valid_mid(bot.get("id")), "Telegram did not return the bot identity.")
    member = api.call("getChatMember", {"chat_id": CHAT_ID, "user_id": bot["id"]})
    require(isinstance(member, dict) and (member.get("status") == "creator" or
            (member.get("status") == "administrator" and member.get("can_post_messages") is True)),
            "Give this bot the channel administrator permission to post messages.")
    return bot

class PhotoArchive:
    def __init__(self, path):
        self.zip = zipfile.ZipFile(path)
        try:
            names = [p for p in self.zip.namelist() if PurePosixPath(p).name == "result.json"]
            require(len(names) == 1, "The source ZIP must contain exactly one result.json.")
            self.root = names[0][:-len("result.json")]
            raw = self.zip.read(names[0])
            git_sha = hashlib.sha1(b"blob " + str(len(raw)).encode() + b"\0" + raw).hexdigest()
            require(git_sha == JSON_BLOB, "The source ZIP does not match the pinned South Movie export.")
            original = {str(m["id"]): m for m in json.loads(raw)["messages"] if m.get("type") == "message"}
            for sid, p in POSTS.items():
                m = original[sid]
                require(m.get("photo") == p["path"] and source_links(m) == WEB_URL.findall(p["caption"]),
                        f"Source photo or movie links differ at source {sid}.")
                self.read(p)
        except BaseException:
            self.zip.close()
            raise

    def read(self, p):
        data = self.zip.read(self.root + p["path"])
        require(len(data) == p["bytes"] and sha(data) == p["sha256"], f"Photo checksum failed at source {p['source_id']}.")
        return data

    def close(self):
        self.zip.close()

def open_source():
    import requests
    cache_file = CACHE / ("south_movie_" + REF[:12] + ".zip")
    if SOURCE_ZIP.strip():
        candidates = [Path(SOURCE_ZIP)]
    else:
        filenames = (cache_file.name, "south-movie-" + REF + ".zip")
        candidates = [WORK / n for n in filenames if (WORK / n).is_file()]
        if INPUT.exists():
            for n in filenames:
                candidates.extend(sorted(INPUT.rglob(n)))
        if cache_file.exists():
            candidates.append(cache_file)
    for f in candidates:
        try:
            media = PhotoArchive(f)
            print(f"Verified {len(POSTS):,} original photos from {f.name}.", flush=True)
            return media
        except (OSError, KeyError, zipfile.BadZipFile, Pause) as exc:
            if SOURCE_ZIP.strip():
                raise Pause(f"Selected source ZIP failed verification: {exc}") from None
            print(f"Skipping unusable cached copy {f.name}.", flush=True)
    CACHE.mkdir(parents=True, exist_ok=True)
    url = f"https://codeload.github.com/{REPO}/zip/{REF}"
    print("Downloading the 413 MiB pinned source archive. All photos are checked before the first channel change.", flush=True)
    for attempt in range(3):
        with tempfile.NamedTemporaryFile(dir=CACHE, suffix=".part", delete=False) as output:
            tmp = Path(output.name)
        try:
            with requests.get(url, stream=True, timeout=(20, 120)) as resp, tmp.open("wb") as out:
                resp.raise_for_status()
                for block in resp.iter_content(1024 * 1024):
                    if block:
                        out.write(block)
            verified = PhotoArchive(tmp)
            try:
                os.replace(tmp, cache_file)
            except BaseException:
                verified.close()
                raise
            return verified
        except (requests.RequestException, OSError, KeyError, zipfile.BadZipFile, Pause) as exc:
            if attempt == 2:
                raise Pause(f"Source download or verification failed ({type(exc).__name__}). Enable Internet, "
                            "or point SOURCE_ZIP to the pinned GitHub ZIP attached as input.") from None
            time.sleep(2 ** attempt)
        finally:
            tmp.unlink(missing_ok=True)

@contextmanager
def single_writer():
    import fcntl
    with (WORK / ".south_movie.lock").open("a+") as lock:
        try:
            fcntl.flock(lock, fcntl.LOCK_EX | fcntl.LOCK_NB)
        except BlockingIOError:
            raise Pause("Another uploader is already using this output folder.") from None
        try:
            yield
        finally:
            fcntl.flock(lock, fcntl.LOCK_UN)

In [ ]:
def load_progress():
    named = [(p.name, normalize(read_json(p))) for p in checkpoint_files()]
    # A later checkpoint carries the newest uncertain attempt. Prefer it to a snapshot.
    if DESTINATION_EXPORT.strip() or not named:
        files = [Path(DESTINATION_EXPORT)] if DESTINATION_EXPORT.strip() else sorted(INPUT.rglob("result.json")) if INPUT.is_dir() else []
        matching = []
        for path in files:
            data = read_json(path)
            if isinstance(data, dict) and str(data.get("id")) in (str(CHAT_ID), str(abs(CHAT_ID) - 10**12)):
                matching.append((path.name, data))
        require(len(matching) <= 1, "Multiple destination exports found. Set DESTINATION_EXPORT to the current file.")
        if DESTINATION_EXPORT.strip():
            require(matching, "The selected export must be the current @southdhamal destination.")
        if matching:
            name, data = matching[0]
            bot_id = named[0][1]["bot_id"] if named else None
            named.append((name, state_from_export(data, bot_id)))
    s = choose_state(named)
    prefix_ok = all(s["done"].get(sid) == "caption" and sid in s["receipts"] for sid in list(POSTS)[:BASELINE_COUNT])
    require(prefix_ok, "This checkpoint predates the pause after 1,227 movie photos. Attach the Output from that run.")
    require(INTERRUPTED_SOURCE in s["receipts"] or INTERRUPTED_ATTEMPT in s["resolutions"] or
            (s["pending"] and s["pending"]["sid"] == INTERRUPTED_SOURCE and s["pending"]["attempt"] == INTERRUPTED_ATTEMPT),
            "The checkpoint lacks evidence of the logged attempt at Annapoorani. Use the latest paused Output.")
    figures = stats(s)
    next_sid = next((sid for sid in POSTS if sid not in s["receipts"]), None)
    print(f"Confirmed movie photos: {figures['confirmed_movie_photos']:,} / {len(POSTS):,}.", flush=True)
    if next_sid:
        print(f"Next outstanding photo: source {next_sid} • {POSTS[next_sid]['caption'].splitlines()[0]}", flush=True)
    return s

def store_issue(s, sid, method, why):
    s["issues"][sid] = {"operation": method, "reason": str(why)[:300]}
    persist(s)

def repair(s, api):
    for sid, row in list(s["receipts"].items()):
        if sid in s["done"] or (sid in s["issues"] and not RETRY_REVIEW_ITEMS):
            continue
        method = "deleteMessage" if row["kind"] == "text" else "editMessageCaption"
        data = {"chat_id": CHAT_ID, "message_id": row["message_id"]}
        if method == "editMessageCaption":
            # An old promo-only photo keeps its original image; clear its obsolete caption.
            data["caption"] = POSTS[sid]["caption"] if sid in POSTS else ""
        try:
            api.call(method, data)
        except Rejected as exc:
            desc = exc.description.lower()
            already_done = ((method == "deleteMessage" and "message to delete not found" in desc)
                            or (method == "editMessageCaption" and "message is not modified" in desc))
            individual = ("message can't be deleted", "message can not be deleted", "message cannot be deleted",
                          "message to edit not found", "message can't be edited", "message can not be edited",
                          "message cannot be edited", "message_id_invalid")
            if not already_done:
                if exc.code == 400 and any(t in desc for t in individual):
                    store_issue(s, sid, method, exc.description)
                    continue
                raise Pause(f"Repair paused on source {sid} at destination message {row['message_id']}. {exc}") from None
        s["issues"].pop(sid, None)
        s["done"][sid] = "deleted" if method == "deleteMessage" else "caption"
        persist(s)
        if len(s["done"]) % 50 == 0:
            print(f"{len(s['done'])} prior messages processed.", flush=True)

def send_remaining(s, api, media):
    for sid, p in POSTS.items():
        if sid in s["receipts"] or (sid in s["issues"] and not RETRY_REVIEW_ITEMS):
            continue
        photo = media.read(p)
        # Persist intent before the request. A kernel interruption leaves an inspectable pending entry.
        s["issues"].pop(sid, None)
        s["pending"] = {"sid": sid, "kind": "photo", "attempt": uuid.uuid4().hex}
        persist(s)
        try:
            result = api.call("sendPhoto", {"chat_id": CHAT_ID, "caption": p["caption"], "disable_notification": "true"},
                              photo=(PurePosixPath(p["path"]).name, photo))
        except Rejected as exc:
            # A structured 4xx is a definitive rejection; a timeout or 502 takes the separate uncertain path.
            attempt_id = s["pending"]["attempt"]
            s["resolutions"][attempt_id] = {"sid": sid, "message_id": 0}
            s["pending"] = None
            desc = exc.description.lower()
            if exc.code == 400 and any(t in desc for t in ("photo_invalid_dimensions", "image_process_failed",
                                                            "file is too big", "photo must be non-empty")):
                store_issue(s, sid, "sendPhoto", exc.description)
                continue
            persist(s)
            raise Pause(f"Telegram rejected source {sid}. {exc} Resume from the saved checkpoint.") from None
        except UnknownSend as exc:
            raise Pause(f"{exc} Source {sid}:0 is saved as pending. Inspect the channel before resuming.") from None
        if not (isinstance(result, dict) and valid_mid(result.get("message_id"))
                and isinstance(result.get("chat"), dict) and result["chat"].get("id") == CHAT_ID
                and result.get("photo")):
            raise Pause(f"Source {sid}:0 returned an incomplete photo receipt. Check the channel before resuming.")
        mid = result["message_id"]
        require(all(row["message_id"] != mid for row in s["receipts"].values()),
                f"Telegram returned an already-recorded message ID for {sid}:0. Review the pending delivery.")
        add_receipt(s, sid, mid, "photo")
        s["done"][sid] = "caption"
        s["pending"] = None
        persist(s)
        if len(s["receipts"]) % 25 == 0:
            print(f"{sum(k in s['receipts'] for k in POSTS):,}/{len(POSTS):,} movie photos confirmed.", flush=True)

def stats(s):
    confirmed = len(POSTS.keys() & s["receipts"].keys())
    return {"confirmed_movie_photos": confirmed, "planned_movie_photos": len(POSTS),
            "remaining_movie_photos": len(POSTS) - confirmed,
            "confirmed_clean_movie_captions": len(POSTS.keys() & s["done"].keys()),
            "old_text_messages_removed": sum(v == "deleted" for v in s["done"].values()),
            "old_nonmovie_photo_captions_cleared": sum(k not in POSTS and v == "caption" for k, v in s["done"].items()),
            "manual_review_items": len(s["issues"]), "pending": s["pending"]}

def report(status, s=None, reason=""):
    result = {"status": status, "reason": reason, "target": TARGET, "source": REF, **PLAN_COUNTS}
    if s is not None:
        result.update(stats(s))
        review = []
        for sid, issue in s["issues"].items():
            msg_id = s["receipts"].get(sid, {}).get("message_id")
            review.append({"source_id": int(sid), **issue, "destination_message_id": msg_id,
                           "destination_post": f"https://t.me/{TARGET[1:]}/{msg_id}" if msg_id else None})
        if RUN_MODE == "REPAIR_AND_CONTINUE":
            write_json(WORK / "south_movie_repair_review.json", review)
    write_json(WORK / "south_movie_run_summary.json", result)
    print("\n" + status + (" • " + reason if reason else ""), flush=True)
    if s is not None:
        print(json.dumps(stats(s), indent=2), flush=True)
    print("Output folder:", WORK)
    return result

def execute():
    s = None
    try:
        require(RUN_MODE in ("PREVIEW", "REPAIR_AND_CONTINUE"), "Set RUN_MODE to PREVIEW or REPAIR_AND_CONTINUE.")
        if RUN_MODE == "PREVIEW":
            s = load_progress()
            previous_text = sum(row["kind"] == "text" for row in s["receipts"].values())
            print(f"{len(s['receipts']) - previous_text} saved photos • {previous_text} saved standalone text messages.")
            if s["pending"]:
                print("Recovery key for this attempt:", s["pending"]["attempt"])
                sid = s["pending"]["sid"]
                print(f"Pending source {sid}:0. " + (POSTS[sid]["caption"] if sid in POSTS else "Inspect the destination message."))
            return report("PREVIEW", s, "Progress checked. Telegram was not contacted.")
        with single_writer():
            s = load_progress()
            import requests
            with requests.Session() as session:
                api = Telegram(bot_token(), session)
                bot = verify_target(api)
                if s["bot_id"] is None:
                    require(str(bot.get("username", "")).lower() == LOG_BOT,
                            "Destination-export recovery requires the original South Movie bot.")
                    s["bot_id"] = bot["id"]
                else:
                    require(s["bot_id"] == bot["id"], "The bot differs from the one in the checkpoint.")
                persist(s)
                resolve_pending(s)
                need_media = any(k not in s["receipts"] and (k not in s["issues"] or RETRY_REVIEW_ITEMS) for k in POSTS)
                media = open_source() if need_media else None
                try:
                    repair(s, api)
                    if media is not None:
                        send_remaining(s, api, media)
                finally:
                    if media is not None:
                        media.close()
                figures = stats(s)
                status = "COMPLETE" if (not s["issues"] and not s["pending"]
                        and figures["remaining_movie_photos"] == 0
                        and figures["confirmed_clean_movie_captions"] == len(POSTS)) else "MANUAL REVIEW"
                return report(status, s)
    except (Pause, Rejected, OSError, zipfile.BadZipFile, KeyError, ValueError, TypeError) as exc:
        # A normal end lets Kaggle retain Output even after a flood limit or ambiguous send.
        return report("PAUSED", s, str(exc))

RUN_RESULT = execute()

### Output

**COMPLETE** means all planned photos and captions are confirmed. **PAUSED** gives the cause and any uncertain photo's source ID and recovery key. **MANUAL REVIEW** lists individual Telegram errors. Carry the newest `south_movie_verified_checkpoint.json` into a later Kaggle session. The summary is in `south_movie_run_summary.json`.